# episodes

> Study B of the Phase 8 benchmark: teams play sequences of episodes on new worlds, with an exploration level that each agent learns across episodes (with or without discovery credit) and with or without persisted map state.

Spec: `specs/multiagent/phase8/` (design-low-level section 5).

In [ ]:
#| default_exp episodes

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.bench_metrics import discoveries, freeRiding
from explRL.credit import Credit
from explRL.harness import Blackboard
from explRL.harness_agents import CommBandit
from explRL.harness_runs import benchWorld, netHarvest
from explRL.multi import runTeam
from explRL.team_refs import ledgerArrays

LEVELS = (0.5, 1.0, 2.0, 4.0)
ARMS = ('fixed-0.5', 'fixed-1', 'fixed-2', 'fixed-4', 'strategy', 'all')


class LevelBandit(CommBandit):
    """`CommBandit` whose exploration level (`cUCB`) is fixed within an episode and chosen across episodes: round
    robin (offset by agent) for the first `len(levels)` episodes, then UCB over levels valued by episode reward plus
    discovery credit per second. `fixed` pins the level."""
    def __init__(self, levels=LEVELS, cLevel=0.1, fixed=None, w=1.0, K=None, H=10.0):
        super().__init__(None, w, K, 1.0, H)
        self.levels, self.cLevel, self.fixed = tuple(levels), float(cLevel), fixed
        self.S, self.n = np.zeros(len(self.levels)), np.zeros(len(self.levels), int)
        self.e, self.k, self.credit, self.epReward = 0, 0, None, 0.0
        self.level = self.prevLevel = None

    def beginEpisode(self, channel, credit):
        self.channel, self.credit, self.epReward, self.prevLevel = channel, credit, 0.0, self.level
        if self.fixed is not None:
            self.level = float(self.fixed)
        else:
            E = len(self.levels)
            if self.e < E:
                self.k = (self.e + self.id) % E
            else:
                self.k = int(np.argmax(self.S / self.n + self.cLevel * np.sqrt(np.log1p(self.e) / self.n)))
            self.level = self.levels[self.k]
        self.cUCB = self.level

    def _takeCredit(self):
        for _, c in (self.credit.take(self.id) if self.credit is not None else ()):
            self.epReward += c

    def event(self, world, t):
        self._takeCredit()
        return super().event(world, t)

    def learn(self, obs, target, reward, info):
        super().learn(obs, target, reward, info)
        self.epReward += reward
        if self.credit is not None and info['kind'] == 'stay':
            self.credit.stay(self._world, self.id, obs['location'], info['harvest'], self.k)

    def endEpisode(self, T):
        self._takeCredit()
        if self.fixed is None:
            self.S[self.k] += self.epReward / T
            self.n[self.k] += 1
        self.e += 1

    def reset(self, keep):
        "'none': forget everything; 'strategy': forget the map (own stays, reports), keep level values; 'all': keep all."
        if keep != 'all':
            self.rates, self.reports = None, None
        if keep == 'none':
            self.S[:], self.n[:], self.e = 0.0, 0, 0


def episodeRow(arm, beta, N, seed, e, world, hist, members, credit, T):
    L = ledgerArrays(world.ledger)
    lv = np.array([m.level for m in members])
    prev = [m.prevLevel for m in members]
    disc = discoveries(world, L)
    return dict(study='studyB', factor='studyB', level=f'{arm}|beta={beta}', arm=arm, beta=float(beta), seed=seed,
                N=N, episode=e, Trun=T, channel='blackboard', budget='episode',
                perAgentSecond=float(netHarvest(world, L).sum()) / (N * T), levelMean=float(lv.mean()),
                levelStd=float(lv.std()),
                levelStable=float('nan') if None in prev else float(np.mean([m.level == m.prevLevel for m in members])),
                discoveries=int(disc.sum()), freeRiding=freeRiding(L, disc, N), events=len(hist),
                credited=float(sum(s['credited'] for s in credit.stats.values())) if credit is not None else 0.0)


def runEpisodes(arm, N, seed, episodes=12, T=600.0, beta=0.0, lam=0.01, cLevel=0.1):
    "Play one study B sequence; world seed 10 000 + 100 seed + e for episode e. Returns one row per episode."
    if arm not in ARMS:
        raise ValueError(f'unknown arm {arm}')
    keep = arm if arm in ('strategy', 'all') else 'none'
    fixed = float(arm.split('-')[1]) if arm.startswith('fixed') else None
    members = [LevelBandit(cLevel=cLevel, fixed=fixed) for _ in range(N)]
    for i, m in enumerate(members):
        m.id = i
    board, rows = None, []
    for e in range(episodes):
        world = benchWorld()
        world.reset(10_000 + 100 * seed + e, N)
        if keep == 'all' and board is not None:
            board.rebase(T)
        else:
            board = Blackboard(world.L, None, lam, N)
        credit = Credit(N, beta) if beta > 0 else None
        for m in members:
            if e:
                m.reset(keep)
            m.beginEpisode(board, credit)
        hist = runTeam(members, world, T)
        for m in members:
            m.endEpisode(T)
        rows.append(episodeRow(arm, beta, N, seed, e, world, hist, members, credit, T))
    return rows

## Tests

In [ ]:
import json
import numpy as np
from collections import deque
from fastcore.test import test_eq, test_close
import explRL.episodes as ep
from explRL.episodes import LevelBandit, runEpisodes, LEVELS
from explRL.credit import Credit
from explRL.harness import Blackboard

# round robin with an agent offset for the first len(levels) episodes, then UCB over level values
ms = [LevelBandit() for _ in range(4)]
for i, m in enumerate(ms): m.id = i
for e in range(4):
    for m in ms: m.beginEpisode(None, None)
    test_eq([m.level for m in ms], [LEVELS[(e + i) % 4] for i in range(4)])
    for m in ms: m.epReward = 60.0 * (1 + m.k == 2); m.endEpisode(60.0)     # level index 1 earned 1/s, the rest 0
m = ms[0]; m.beginEpisode(None, None); test_eq(m.level, LEVELS[1])
m.cLevel = 100.0; m.n[:] = (9, 9, 1, 9); m.beginEpisode(None, None); test_eq(m.level, LEVELS[2])   # big bonus: least used
# credit received adds to the episode's value
cr = Credit(1, beta=1.0); lb = LevelBandit(); lb.id = 0; lb.beginEpisode(None, cr)
cr.inbox[0].append((0, 2.5)); lb._takeCredit(); test_close(lb.epReward, 2.5)
# reset: 'strategy' forgets the map and keeps level values; 'all' keeps both; 'none' forgets both
for keep, mapKept, valuesKept in (('strategy', False, True), ('all', True, True), ('none', False, False)):
    lb = LevelBandit(); lb.rates, lb.reports = [deque([1.0])], [deque([2.0])]; lb.S[:], lb.n[:], lb.e = 1.0, 1, 4
    lb.reset(keep)
    test_eq(lb.rates is not None, mapKept); test_eq(lb.reports is not None, mapKept)
    test_eq((lb.n.sum() > 0, lb.e > 0), (valuesKept, valuesKept))
# fixed arms never change level, so their levelStable is 1 after the first episode; runs are deterministic
rows = runEpisodes('fixed-2', 4, 1000, episodes=3, T=40.0)
test_eq([r['levelMean'] for r in rows], [2.0] * 3); test_eq([r['levelStable'] for r in rows][1:], [1.0, 1.0])
assert np.isnan(rows[0]['levelStable'])
a = runEpisodes('strategy', 4, 1000, episodes=5, T=40.0, beta=1.0)
test_eq(json.dumps(a), json.dumps(runEpisodes('strategy', 4, 1000, episodes=5, T=40.0, beta=1.0)))   # nan-safe
test_eq([r['episode'] for r in a], list(range(5)))
test_eq([r['levelMean'] for r in a][:4], [np.mean(LEVELS)] * 4)            # round robin: every level once per episode
assert sum(r['discoveries'] for r in a) > 0
# 'all' keeps one blackboard across episodes (rebased); the other arms build a new one each episode
made = []
class CountingBoard(Blackboard):
    def __init__(self, *a, **k): made.append(1); super().__init__(*a, **k)
ep.Blackboard = CountingBoard
for arm, boards in (('all', 1), ('strategy', 3), ('fixed-1', 3)):
    made.clear(); runEpisodes(arm, 4, 1000, episodes=3, T=40.0); test_eq(len(made), boards)
ep.Blackboard = Blackboard
try:
    runEpisodes('bogus', 2, 0)
except ValueError:
    pass
else:
    raise AssertionError('expected ValueError')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()